# How to read a QML paper
**Quantum Machine Learning · Week 4 (QML for research) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qml/w4d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**How to read a QML paper: reproduce a claim.** Havlíček and coauthors (Nature, 2019) showed a quantum kernel
classifying data that a classical kernel struggles with. Rebuild their "ad hoc" dataset at small scale, check the
claim with this tutorial's protocol, then ask the questions a careful reader would.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, random_unitary
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

def zz_map(x, reps=2):
    """The ZZ feature map of weeks 1 and 2, on two qubits."""
    qc = QuantumCircuit(2)
    for _ in range(reps):
        qc.h([0, 1])
        qc.p(2 * x[0], 0)
        qc.p(2 * x[1], 1)
        qc.cx(0, 1)
        qc.p(2 * (np.pi - x[0]) * (np.pi - x[1]), 1)
        qc.cx(0, 1)
    return qc

# The paper's recipe: label a point by the parity Z⊗Z of its encoded state after a fixed random unitary V,
# and drop points whose value is within a gap of 0.3 of zero.
V = random_unitary(4, seed=7).data
parity = V.conj().T @ np.kron(np.diag([1, -1]), np.diag([1, -1])) @ V

def label_value(x):
    s = Statevector(zz_map(x)).data
    return np.real(s.conj() @ parity @ s)

def ad_hoc(n_per_class, seed, gap=0.3):
    rng = np.random.default_rng(seed)
    X, y = [], []
    while min(y.count(0), y.count(1)) < n_per_class:
        x = rng.uniform(0, 2 * np.pi, 2)
        v = label_value(x)
        c = int(v > 0)
        if abs(v) >= gap and y.count(c) < n_per_class:
            X.append(x)
            y.append(c)
    return np.array(X), np.array(y)

X_all, y_all = ad_hoc(400, seed=0)                     # 800 points, half of each class
fig, ax = plt.subplots(figsize=(4.6, 4.4), layout="constrained")
ax.scatter(X_all[:, 0], X_all[:, 1], c=np.where(y_all == 0, "#ff8a50", "#5577f5"), s=10)
ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_title("The ad hoc dataset")
ax.set_aspect("equal")
plt.show()

## The claim, under this tutorial's protocol
Train on $n$ points, test on 400 others, five random draws per size; $C$ (and the RBF's $\gamma$) tuned by
cross-validation on the training points only.

In [ ]:
def kernel_matrix(fmap, A, B):
    SA = np.array([Statevector(fmap(a)).data for a in A])
    SB = np.array([Statevector(fmap(b)).data for b in B])
    return np.abs(SA.conj() @ SB.T) ** 2

K_all = kernel_matrix(zz_map, X_all, X_all)

def compare(K, sizes=(20, 40, 80, 160, 320), repeats=5):
    rows = {}
    for n in sizes:
        q, c = [], []
        for seed in range(repeats):
            idx = np.random.default_rng(seed).permutation(len(X_all))
            tr, te = idx[:n], idx[n:n + 400]
            folds = min(5, np.bincount(y_all[tr], minlength=2).min())
            qs = GridSearchCV(SVC(kernel="precomputed"), {"C": [0.1, 1, 10, 100, 1000]}, cv=folds).fit(K[np.ix_(tr, tr)], y_all[tr])
            q.append(qs.score(K[np.ix_(te, tr)], y_all[te]))
            cs = GridSearchCV(SVC(kernel="rbf"), {"C": [0.1, 1, 10, 100, 1000], "gamma": [0.01, 0.1, 1, 10]}, cv=folds).fit(X_all[tr], y_all[tr])
            c.append(cs.score(X_all[te], y_all[te]))
        rows[n] = (np.mean(q), np.std(q), np.mean(c), np.std(c))
        print(f"{n:3d} training points: quantum kernel {np.mean(q):.3f} ± {np.std(q):.3f}   tuned RBF {np.mean(c):.3f} ± {np.std(c):.3f}")
    return rows

print("guessing the majority class: 0.500")
results = compare(K_all)

## Your turn: was it the quantumness, or the match?
The ZZ kernel won on data generated from the ZZ feature map. Swap in a different quantum feature map, the ZZ map
with one layer instead of two, or the angle map, and rerun the comparison. How much of the edge survives?

In [ ]:
def angle_map(x):
    qc = QuantumCircuit(2)
    qc.ry(x[0], 0)
    qc.ry(x[1], 1)
    return qc

# TODO: build kernel_matrix(lambda x: zz_map(x, reps=1), X_all, X_all) and kernel_matrix(angle_map, X_all, X_all),
#       and run compare(...) with each

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def angle_map(x):
    qc = QuantumCircuit(2)
    qc.ry(x[0], 0)
    qc.ry(x[1], 1)
    return qc

for name, fmap in (("ZZ map, 1 layer", lambda x: zz_map(x, reps=1)), ("angle map", angle_map)):
    print(name)
    compare(kernel_matrix(fmap, X_all, X_all), sizes=(40, 160, 320))
print("The one-layer ZZ map, a close relative of the map that made the labels, keeps part of the edge with few points")
print("and falls to the RBF's level with more; the unrelated angle map is worse than the RBF at every size.")
print("The advantage came from matching how the labels were made, not from quantumness as such.")